# Entrenamiento de los cultivos principales

Selecciona cultivos por superficie cosechada con cobertura suficiente y entrena un modelo equivalente a `v1` para cada uno usando clima, suelo e historial productivo calculado solo con anos anteriores. Esta etapa construye los modelos candidatos para la futura matriz municipio-cultivo; todavia no realiza la redistribucion.

In [1]:

import pandas as pd
from sklearn.base import clone

from proyecto_agronomia.config import INTERIM_DATA_DIR, PROCESSED_DATA_DIR
from modeling.cultivos import (
    VARIABLES_CATEGORICAS, VARIABLES_NUMERICAS, catalogar_cultivos,
    cargar_registros_cultivos, preparar_cultivo, seleccionar_principales, separar_xy,
)
from modeling.registry import guardar_modelo, ruta_version_modelo
from modeling.train import crear_particiones, seleccionar_modelo
from modeling.train_original import comparar_modelos_originales

2026-11-21 09:00:25.819 | INFO     | proyecto_agronomia.config:<module>:15 - PROJ_ROOT path is: /home/dusty/Documents/Repositories/proyecto_agronomia


/home/dusty/Documents/Repositories/proyecto_agronomia


In [ ]:
RUTA_DATOS = PROCESSED_DATA_DIR / 'agricultura_clima_edafologia_historial_2003_2025.csv'
CANTIDAD_CULTIVOS = 385  # maximo; los filtros determinan cuantos son elegibles
UNIDAD = 'Tonelada'
MINIMO_ANIOS = 15
MINIMO_MUNICIPIOS = 100
ANIO_INICIO_PRUEBA = 2022
PROPORCION_MUNICIPIOS_RESERVADOS = 0.20
SEMILLA = 42
VERSION = 'v1_principales'
ENTRENAR = True

if not RUTA_DATOS.is_file():
    raise FileNotFoundError(f'No se encontro el dataset de entrenamiento: {RUTA_DATOS}')
print(f'Dataset de entrenamiento: {RUTA_DATOS}')

## Catálogo y selección transparente

In [ ]:
catalogo = catalogar_cultivos(ruta=RUTA_DATOS, unidad=UNIDAD)
principales = seleccionar_principales(
    catalogo, cantidad=CANTIDAD_CULTIVOS, minimo_anios=MINIMO_ANIOS,
    minimo_municipios=MINIMO_MUNICIPIOS,
)
principales.insert(0, 'posicion', range(1, len(principales) + 1))
display(principales)
assert 0 < len(principales) <= CANTIDAD_CULTIVOS
assert principales['anios'].ge(MINIMO_ANIOS).all()
assert principales['municipios'].ge(MINIMO_MUNICIPIOS).all()
print(f'Cultivos elegibles: {len(principales):,} de hasta {CANTIDAD_CULTIVOS:,} solicitados')

## Cargar una sola vez los registros seleccionados

La lectura por bloques evita cargar los 825 mil registros completos y evita releer el archivo para cada cultivo.

In [ ]:
nombres_cultivos = principales['Nomcultivo'].tolist()
datos_principales = cargar_registros_cultivos(
    nombres_cultivos, ruta=RUTA_DATOS, unidad=UNIDAD,
)
display(datos_principales.groupby('Nomcultivo').agg(
    registros=('cvegeo', 'size'), municipios=('cvegeo', 'nunique'),
    anio_inicial=('Anio', 'min'), anio_final=('Anio', 'max'),
).sort_values('registros', ascending=False))

## Entrenamiento reanudable

Cada modelo terminado se guarda inmediatamente. Si se interrumpe la ejecución, al correr de nuevo se omiten los artefactos ya existentes.

In [ ]:
resumen_modelos = []
if ENTRENAR:
    for numero, cultivo in enumerate(nombres_cultivos, start=1):
        destino = ruta_version_modelo(cultivo, VERSION)
        if (destino / 'model.joblib').is_file():
            print(f'[{numero}/{len(nombres_cultivos)}] Omitido, ya existe: {cultivo}')
            continue
        print(f'[{numero}/{len(nombres_cultivos)}] Entrenando: {cultivo}')
        datos = preparar_cultivo(datos_principales, cultivo)
        particiones = crear_particiones(
            datos, anio_prueba=ANIO_INICIO_PRUEBA,
            proporcion_municipios=PROPORCION_MUNICIPIOS_RESERVADOS, semilla=SEMILLA,
        )
        entrenados, metricas = comparar_modelos_originales(datos, particiones, SEMILLA)
        ganador = seleccionar_modelo(metricas)
        X, y = separar_xy(datos)
        modelo_final = clone(entrenados[ganador]).fit(X, y)
        guardar_modelo(
            modelo=modelo_final, cultivo=cultivo, version=VERSION,
            metadata={
                'algoritmo': ganador, 'unidad': UNIDAD,
                'periodo_datos': f"{datos['Anio'].min()}-{datos['Anio'].max()}",
                'inicio_prueba': ANIO_INICIO_PRUEBA,
                'proporcion_municipios_reservados': PROPORCION_MUNICIPIOS_RESERVADOS,
                'semilla': SEMILLA, 'registros': len(datos),
                'municipios': datos['cvegeo'].nunique(), 'objetivo': 'Rendimiento',
                'seleccion': 'cultivos principales por superficie cosechada',
            },
            metricas={'comparacion_modelos': metricas.to_dict(orient='records')},
            variables=[*VARIABLES_NUMERICAS, *VARIABLES_CATEGORICAS],
        )
        mejores = metricas.loc[metricas['modelo'].eq(ganador)]
        for fila in mejores.to_dict(orient='records'):
            resumen_modelos.append({'cultivo': cultivo, 'ganador': ganador, **fila})
        print(f'  Guardado {ganador} en {destino}')
else:
    print('Entrenamiento omitido.')

## Resumen de modelos entrenados en esta ejecución

In [ ]:
df_resumen_modelos = pd.DataFrame(resumen_modelos)
if not df_resumen_modelos.empty:
    display(df_resumen_modelos.sort_values(['particion', 'rmse']))
    destino_resumen = INTERIM_DATA_DIR / 'modeling' / 'cultivos_principales_metricas.csv'
    destino_resumen.parent.mkdir(parents=True, exist_ok=True)
    df_resumen_modelos.to_csv(destino_resumen, index=False)
    print(f'Resumen guardado: {destino_resumen}')
else:
    print('No hubo modelos nuevos; los existentes fueron respetados.')